# Silo-model: wij-zij-denken tussen engineers en managers

Agent-based model van hoe vertrouwen, groepsidentificatie en silo's ontstaan in een projectorganisatie, en welke interventies helpen.

**Zo gebruik je dit notebook in Google Colab**
1. Vul in de eerste codecel je GitHub-gebruikersnaam in.
2. Kies *Runtime → Run all* (of druk op ▶ bij elke cel).
3. Verander daarna gerust parameters in deel 2 en draai de cel opnieuw.

In [ ]:
# --- Installatie (alleen nodig in Colab) ---
import os

GITHUB_GEBRUIKER = "JOUW-GEBRUIKERSNAAM"   # <- vul hier je GitHub-naam in
REPO = f"https://github.com/{GITHUB_GEBRUIKER}/silo-model.git"

if not os.path.exists("silo_model.py"):
    if not os.path.exists("silo-model"):
        !git clone {REPO}
    %cd silo-model

try:
    import mesa
except ImportError:
    %pip install -q -r requirements.txt

## 1. Eén run van het model

Elke tick is één werkdag. We draaien 250 ticks (ongeveer een werkjaar) met de standaardwaarden.

In [ ]:
import matplotlib.pyplot as plt
from silo_model import run_model, run_many, final_values

data, model = run_model(steps=250, seed=1)
data.iloc[[0, 49, 99, 249]].round(3).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
data["E-I index"].plot(ax=axes[0], title="Silo-index (E-I)")
axes[0].axhline(model.ei_random, linestyle="--", color="grey", label="willekeurig contact")
axes[0].legend()
data[["Vertrouwen engineers", "Vertrouwen managers"]].plot(ax=axes[1], title="Vertrouwen in de andere groep")
axes[1].set_ylim(0, 1)
plt.show()

## 2. Zelf experimenteren

Verander hieronder één of meer parameters en vergelijk met de basis. Een paar om mee te beginnen:

| Parameter | Standaard | Betekenis |
|---|---|---|
| `base_success` | 0.7 | basiskans dat een gesprek met de andere groep goed gaat |
| `negativity_bias` | 2.0 | hoeveel zwaarder een slechte ervaring telt |
| `homophily` | 1.0 | hoe sterk identificatie de partnerkeuze stuurt |
| `gossip` | 0.05 | hoeveel vertrouwen je overneemt van groepsgenoten |
| `status_gap` | 0.3 | hoeveel statusverschil de opbouw van vertrouwen dempt |
| `forced_contact` | 0.2 | aandeel gesprekken dat projectwerk afdwingt |
| `shared_goal` | 0.0 | extra slaagkans door een gedeeld projectdoel |
| `n_bridges` | 0 | aantal engineers dat brugfiguur wordt |

Alle parameters staan met uitleg bovenin `SiloModel` in `silo_model.py`.

In [ ]:
mijn_parameters = dict(negativity_bias=1.0)   # <- pas dit aan

basis = run_many(runs=10)
variant = run_many(runs=10, **mijn_parameters)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for naam, d in (("basis", basis), ("variant", variant)):
    gem = d.groupby("tick").mean(numeric_only=True)
    gem["E-I index"].plot(ax=axes[0], label=naam)
    ((gem["Vertrouwen engineers"] + gem["Vertrouwen managers"]) / 2).plot(ax=axes[1], label=naam)
axes[0].set_title("Silo-index (E-I)"); axes[1].set_title("Vertrouwen (gemiddeld)")
axes[1].set_ylim(0, 1)
for ax in axes: ax.legend()
plt.show()

print("Eindwaarden (gemiddelde laatste 25 dagen):")
print("basis  ", final_values(basis)[["E-I index", "Vertrouwen engineers"]].mean().round(3).to_dict())
print("variant", final_values(variant)[["E-I index", "Vertrouwen engineers"]].mean().round(3).to_dict())

## 3. Alle experimenten en grafieken

Dit draait de vier experimenten uit `run_experiments.py` in de snelle versie (minder runs). Zonder `--quick` krijg je de volledige versie; die duurt een paar minuten.

In [ ]:
!python run_experiments.py --quick

In [ ]:
from IPython.display import Image, display
for f in ["01_basisrun", "02_netwerk", "03_interventies", "04_kantelpunt"]:
    display(Image(f"results/{f}.png"))

## Volgende stappen

- Voeg een interventie toe die nog ontbreekt, bijvoorbeeld **rotatie**: wissel elke 50 ticks mensen van projectteam.
- Test of **meer contact averechts werkt** als contact vaak mislukt (Flache & Macy, 2011): vergelijk `forced_contact` bij een lage en hoge `base_success`.
- Vervang de gegokte startwaarden door waarden uit je **interviews en enquête**.